# CDT501 Lecture 10: Python for Finance 金融中的 Python
Run the cells from top to bottom. 从上到下运行。

**Teaching only, not investment advice. No real money, no exchange account. 仅供教学，不构成投资建议，不用真钱，不开账户。**

In [ ]:
!pip -q install yfinance
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import yfinance as yf
from datetime import date, timedelta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dense, Dropout, LSTM, Input
from tensorflow.keras.callbacks import EarlyStopping
print("Ready.")

## Part 1: Risk and reward 第一部分：风险和收益
### Get the data 获取数据

In [ ]:
import json, os, urllib.request

def get_prices(days_ago_start=365, days_ago_end=0):
    """One year of hourly BTC prices. Tries Binance's public data, then Yahoo, then btc_1h.csv."""
    start = date.today() - timedelta(days=days_ago_start)
    end = date.today() - timedelta(days=days_ago_end)
    try:
        rows, t = [], int(pd.Timestamp(start, tz="UTC").timestamp() * 1000)
        stop = int(pd.Timestamp(end, tz="UTC").timestamp() * 1000)
        while t < stop:
            url = f"https://data-api.binance.vision/api/v3/klines?symbol=BTCUSDT&interval=1h&limit=1000&startTime={t}&endTime={stop}"
            part = json.load(urllib.request.urlopen(url, timeout=20))
            if not part: break
            rows += part
            t = part[-1][0] + 3600000
        df = pd.DataFrame(rows).iloc[:, :6].astype(float)
        df.columns = ["time", "Open", "High", "Low", "Close", "Volume"]
        df.index = pd.to_datetime(df["time"], unit="ms", utc=True)
        print("Data from Binance:", len(df), "hours")
        return df.drop(columns="time")
    except Exception as e:
        print("Binance failed:", e)
    try:
        df = yf.Ticker("BTC-USD").history(interval="1h", start=start, end=end)
        if len(df) > 1000:
            print("Data from Yahoo:", len(df), "hours")
            return df
    except Exception as e:
        print("Yahoo failed:", e)
    if os.path.exists("btc_1h.csv"):
        df = pd.read_csv("btc_1h.csv", index_col=0, parse_dates=True)
        print("Data from btc_1h.csv:", len(df), "hours")
        return df
    raise RuntimeError("No data. Upload btc_1h.csv from Moodle (folder icon on the left), then run this cell again.")

prices = get_prices()
prices.tail()

### Return, volatility, Sharpe, drawdown 收益、波动、夏普、回撤

In [ ]:
daily = prices["Close"].resample("1D").last().dropna()
returns = daily.pct_change().dropna()

total_return = daily.iloc[-1] / daily.iloc[0] - 1
volatility = returns.std() * np.sqrt(365)
sharpe = returns.mean() / returns.std() * np.sqrt(365)
drawdown = daily / daily.cummax() - 1

print(f"Total return:          {total_return:8.1%}")
print(f"Volatility (per year): {volatility:8.1%}")
print(f"Sharpe ratio:          {sharpe:8.2f}")
print(f"Maximum drawdown:      {drawdown.min():8.1%}")

fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax[0].plot(daily); ax[0].set_title("BTC price")
ax[1].fill_between(drawdown.index, drawdown, 0, color="tab:red", alpha=.4); ax[1].set_title("Drawdown: how far below the highest price so far")
plt.tight_layout(); plt.show()

### Monte Carlo: many possible futures 蒙特卡罗：很多种可能的未来

In [ ]:
n_simulations = 1000
n_days = 365
mu, sigma = returns.mean(), returns.std()      # measured from the real data above
start_price = daily.iloc[-1]

paths = start_price * np.exp(np.cumsum(np.random.normal(mu, sigma, (n_simulations, n_days)), axis=1))
final = paths[:, -1]

plt.figure(figsize=(10, 4))
plt.plot(paths[:30].T, lw=.8); plt.title("30 possible next years for BTC (Monte Carlo)")
plt.xlabel("days"); plt.ylabel("price"); plt.show()
print(f"Chance of ending below today's price: {(final < start_price).mean():.0%}")
print(f"Worst 5% of paths end below:          {np.quantile(final, .05):,.0f}")
print(f"Best 5% of paths end above:           {np.quantile(final, .95):,.0f}")

## Part 2: Predict Bitcoin with an LSTM 第二部分：用 LSTM 预测比特币
Last year's case study. 去年的案例。

### Prepare the data 准备数据

In [ ]:
# Same data as Part 1 / 和第一部分相同的数据
ticker = "BTC-USD"
model_data = prices.copy()
print(model_data.head())

In [ ]:
# 准备目标变量（收盘价）/ Prepare target variable (Close price)
target_data = model_data["Close"].values.reshape(-1, 1)

# 数据归一化处理 / Data normalization
# 对目标数据进行归一化 / Normalize target data
target_scaler = MinMaxScaler()
target_data_normalized = target_scaler.fit_transform(target_data)

# 准备输入特征 / Prepare input features
input_features = ['Open', 'High', 'Low', 'Volume']
input_data = model_data[input_features].values

# 对输入特征进行归一化 / Normalize input features
input_scaler = MinMaxScaler()
input_data_normalized = input_scaler.fit_transform(input_data)
print(input_data_normalized[:10])

### Build the LSTM model 构建模型

In [ ]:
def build_lstm_model(input_data, output_size, neurons, activ_func='linear',
                    dropout=0.2, loss='mse', optimizer='adam'):
    """
    构建LSTM模型 / Build LSTM model

    参数说明 / Parameters:
    input_data: 输入数据 / Input data
    output_size: 输出维度 / Output dimension
    neurons: LSTM神经元数量 / Number of LSTM neurons
    activ_func: 激活函数 / Activation function
    dropout: Dropout比率 / Dropout rate
    loss: 损失函数 / Loss function
    optimizer: 优化器 / Optimizer
    """
    model = Sequential()
    model.add(Input(shape=(input_data.shape[1], input_data.shape[2])))
    model.add(LSTM(neurons))
    model.add(Dropout(dropout))
    model.add(Dense(units=output_size))
    model.add(Activation(activ_func))
    model.compile(loss=loss, optimizer=optimizer)
    return model

### Prepare training data 准备训练数据

In [ ]:
# 设置超参数 / Set hyperparameters
np.random.seed(245)  # 设置随机种子 / Set random seed
window_len = 10  # 时间窗口长度 / Time window length
lstm_neurons = 50  # LSTM层神经元数量 / Number of LSTM neurons
epochs = 200  # 训练轮数 / Number of training epochs
batch_size = 128  # 批次大小 / Batch size
loss = 'mean_squared_error'  # 损失函数 / Loss function
dropout = 0.24  # Dropout比率 / Dropout rate
optimizer = 'adam'  # 优化器 / Optimizer

def extract_window_data(input_data, target_data, window_len):
    """
    提取时间窗口数据 / Extract time window data

    将数据转换为适合LSTM训练的序列 / Convert data into sequences suitable for LSTM training
    """
    X = []
    y = []
    for i in range(len(input_data) - window_len):
        X.append(input_data[i : i + window_len])
        y.append(target_data[i + window_len])
    return np.array(X), np.array(y)

# 生成训练数据 / Generate training data
X, y = extract_window_data(input_data_normalized, target_data_normalized, window_len)

# 分割训练集和测试集 / Split into training and testing sets
split_ratio = 0.8
split_index = int(split_ratio * len(X))

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

# 重塑数据维度以适应LSTM / Reshape data dimensions for LSTM
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], len(input_features)))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], len(input_features)))

### Train the model 训练模型 (a few minutes 几分钟)

In [ ]:
# 构建模型 / Build model
model = build_lstm_model(X_train, output_size=1, neurons=lstm_neurons,
                        dropout=dropout, loss=loss, optimizer=optimizer)

# Use early stopping to prevent overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=10)

# Train the model
history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size,
                    verbose=2, validation_data=(X_test, y_test), callbacks=[early_stop])
# Save the trained model's weights
file_path = "./LSTM_" + ticker + "_trained.weights.h5"
model.save_weights(file_path)

### Evaluate 评估

In [ ]:
file_path = "./LSTM_" + ticker + "_trained.weights.h5"
model.load_weights(file_path)
# 进行预测 / Make predictions
preds = model.predict(X_test)
y_test = y[split_index:]

# 反归一化预测结果 / Inverse normalize predictions
preds = preds.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

preds_scaled = target_scaler.inverse_transform(preds)
y_test_scaled = target_scaler.inverse_transform(y_test)

# 绘制预测结果 / Plot prediction results
fig = px.line(x=model_data.index[-len(y_test_scaled):],
              y=[y_test_scaled.flatten(), preds_scaled.flatten()])
newnames = {'wide_variable_0': '实际值/Real Values',
            'wide_variable_1': '预测值/Predictions'}
fig.for_each_trace(lambda t: t.update(name=newnames[t.name], legendgroup=newnames[t.name],
          hovertemplate=t.hovertemplate.replace(t.name, newnames[t.name])))
fig.update_layout(
    title=f"{ticker} 价格预测 / Price Prediction",
    xaxis_title="日期/Date",
    yaxis_title=f"{ticker} 价格/Price",
    legend_title=" "
)
fig.show()

# 添加模型评估指标 / Add model evaluation metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math

# 计算RMSE / Calculate RMSE
rmse = math.sqrt(mean_squared_error(y_test_scaled, preds_scaled))
# 计算MAE / Calculate MAE
mae = mean_absolute_error(y_test_scaled, preds_scaled)

print(f"\n模型评估指标 / Model Evaluation Metrics:")
print(f"均方根误差 / Root Mean Square Error (RMSE): {rmse:.2f}")
print(f"平均绝对误差 / Mean Absolute Error (MAE): {mae:.2f}")

### Direction accuracy 方向准确率

In [ ]:
from matplotlib import pyplot as plt
def calculate_direction_accuracy(y_true, y_pred):
    """
    计算预测方向的准确率

    参数:
    y_true: numpy array, 真实值
    y_pred: numpy array, 预测值

    返回:
    dict: 包含方向准确率统计和详细分析的字典
    """
    # 确保输入是1D数组
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()

    # 计算真实值和预测值的变化方向
    true_direction = np.diff(y_true) > 0
    pred_direction = np.diff(y_pred) > 0

    # 计算方向预测的准确率
    correct_directions = np.sum(true_direction == pred_direction)
    total_predictions = len(true_direction)
    accuracy = correct_directions / total_predictions * 100

    # 计算上涨和下跌的分别准确率
    true_up_indices = true_direction == True
    true_down_indices = true_direction == False

    up_accuracy = np.sum((true_direction == pred_direction) & true_up_indices) / np.sum(true_up_indices) * 100
    down_accuracy = np.sum((true_direction == pred_direction) & true_down_indices) / np.sum(true_down_indices) * 100

    # 创建混淆矩阵
    tp = np.sum((true_direction == True) & (pred_direction == True))  # 正确预测上涨
    tn = np.sum((true_direction == False) & (pred_direction == False))  # 正确预测下跌
    fp = np.sum((true_direction == False) & (pred_direction == True))  # 错误预测上涨
    fn = np.sum((true_direction == True) & (pred_direction == False))  # 错误预测下跌

    results = {
        'overall_accuracy': accuracy,
        'up_accuracy': up_accuracy,
        'down_accuracy': down_accuracy,
        'total_predictions': total_predictions,
        'correct_predictions': correct_directions,
        'confusion_matrix': {
            'true_positive': tp,  # 正确预测上涨
            'true_negative': tn,  # 正确预测下跌
            'false_positive': fp,  # 错误预测上涨
            'false_negative': fn   # 错误预测下跌
        },
        'directions': {
            'true_direction': true_direction,
            'pred_direction': pred_direction
        }
    }

    return results

# 使用方法示例：
# 将这段代码放在原始代码的预测部分之后
results = calculate_direction_accuracy(y_test_scaled, preds_scaled)

# 打印详细结果
print("\n=== 方向预测分析 ===")
print(f"总体方向准确率: {results['overall_accuracy']:.2f}%")
print(f"上涨预测准确率: {results['up_accuracy']:.2f}%")
print(f"下跌预测准确率: {results['down_accuracy']:.2f}%")
print(f"总预测次数: {results['total_predictions']}")
print(f"正确预测次数: {results['correct_predictions']}")

print("\n混淆矩阵:")
print(f"正确预测上涨 (True Positive): {results['confusion_matrix']['true_positive']}")
print(f"正确预测下跌 (True Negative): {results['confusion_matrix']['true_negative']}")
print(f"错误预测上涨 (False Positive): {results['confusion_matrix']['false_positive']}")
print(f"错误预测下跌 (False Negative): {results['confusion_matrix']['false_negative']}")

# 可视化方向预测结果
plt.figure(figsize=(14, 7))
plt.subplot(2, 1, 1)
plt.plot(model_data.index[-len(y_test_scaled):], y_test_scaled.flatten(), label='Real Values', color='blue')
plt.plot(model_data.index[-len(preds):], preds.flatten(), label='Predictions', color='orange')
plt.title(f'{ticker} Price Prediction')
plt.legend()
plt.grid(True)

# 添加方向准确性的可视化
plt.subplot(2, 1, 2)
true_direction = results['directions']['true_direction']
pred_direction = results['directions']['pred_direction']
correct_pred = true_direction == pred_direction

plt.scatter(model_data.index[-len(true_direction):][correct_pred],
           [1] * sum(correct_pred),
           color='green',
           label='Correct Direction',
           alpha=0.5)
plt.scatter(model_data.index[-len(true_direction):][~correct_pred],
           [0] * sum(~correct_pred),
           color='red',
           label='Wrong Direction',
           alpha=0.5)
plt.title('Direction Prediction Accuracy')
plt.ylabel('Correct (1) / Wrong (0)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Improved model: add technical indicators 改进模型：加入技术指标
(a few minutes 几分钟)

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dense, Dropout, LSTM, Input
from datetime import date, timedelta

# --- Parameters (replace Streamlit input with direct values)
ticker = "BTC-USD"  # Choose your ticker here (e.g., "ETH-USD", "BTC-USD")

# Same data as Part 1
model_data = prices.copy()

# Extract the 'Close' column for prediction
target_data = model_data["Close"].values.reshape(-1, 1)

# --- Normalize the target data (Close Prices)
target_scaler = MinMaxScaler()
target_data_normalized = target_scaler.fit_transform(target_data)
# Add technical indicators to input_features
model_data['MA_20'] = model_data['Close'].rolling(window=20).mean()
model_data['EMA_20'] = model_data['Close'].ewm(span=20, adjust=False).mean()

# Calculate RSI
delta = model_data['Close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()
rs = avg_gain / avg_loss
model_data['RSI'] = 100 - (100 / (1 + rs))

# Calculate Bollinger Bands
model_data['BB_upper'] = model_data['MA_20'] + (model_data['Close'].rolling(window=20).std() * 2)
model_data['BB_lower'] = model_data['MA_20'] - (model_data['Close'].rolling(window=20).std() * 2)

# Calculate MACD
ema_12 = model_data['Close'].ewm(span=12, adjust=False).mean()
ema_26 = model_data['Close'].ewm(span=26, adjust=False).mean()
model_data['MACD'] = ema_12 - ema_26
# Drop rows with NaN values (caused by rolling calculations)
model_data = model_data.dropna()
# Include these new features
input_features = ['Open', 'High', 'Low', 'Volume', 'MA_20', 'EMA_20', 'RSI', 'BB_upper', 'BB_lower', 'MACD']
input_data = model_data[input_features].values

input_scaler = MinMaxScaler()
input_data_normalized = input_scaler.fit_transform(input_data)



# Function to build the LSTM model
def build_lstm_model(input_data, output_size, neurons, activ_func='linear', dropout=0.2, loss='mse', optimizer='adam'):
    model = Sequential()
    model.add(Input(shape=(input_data.shape[1], input_data.shape[2])))  # Explicit Input layer
    model.add(LSTM(neurons))
    model.add(Dropout(dropout))
    model.add(Dense(units=output_size))
    model.add(Activation(activ_func))
    model.compile(loss=loss, optimizer=optimizer)
    return model

# Hyperparameters
np.random.seed(245)
window_len = 10
lstm_neurons = 50
epochs = 200
batch_size = 128
loss = 'mean_squared_error'
dropout = 0.24
optimizer = 'adam'

# Function to extract window data
def extract_window_data(input_data, target_data, window_len):
    X = []
    y = []
    for i in range(len(input_data) - window_len):
        X.append(input_data[i : i + window_len])
        y.append(target_data[i + window_len])
    return np.array(X), np.array(y)

# Extract window data
X, y = extract_window_data(input_data_normalized, target_data_normalized, window_len)

# Split the data into training and testing sets
split_ratio = 0.8
split_index = int(split_ratio * len(X))

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

# Reshape input to be 3D [samples, time steps, features] as required by LSTM
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], len(input_features)))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], len(input_features)))

# Build and compile the model
model = build_lstm_model(X_train, output_size=1, neurons=lstm_neurons, dropout=dropout, loss=loss, optimizer=optimizer)

# Train the model
history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=2, validation_data=(X_test, y_test))

# Save the trained model's weights
file_path = "./LSTM_" + ticker + "_trained.weights.h5"
model.save_weights(file_path)

# Make predictions using the trained model
preds = model.predict(X_test)
y_test = y[split_index:]

# Inverse normalize the predictions
preds = preds.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

preds_scaled = target_scaler.inverse_transform(preds)
y_test_scaled = target_scaler.inverse_transform(y_test)

# Plot the results
fig = px.line(x=model_data.index[-len(y_test_scaled):],
              y=[y_test_scaled.flatten(), preds_scaled.flatten()])
newnames = {'wide_variable_0': 'Real Values', 'wide_variable_1': 'Predictions'}
fig.for_each_trace(lambda t: t.update(name=newnames[t.name],legendgroup=newnames[t.name],
       hovertemplate=t.hovertemplate.replace(t.name, newnames[t.name])))
fig.update_layout(
    xaxis_title="Date",
    yaxis_title=ticker + " Price",
    legend_title=" "
)
fig.show()

### Direction accuracy of the improved model 改进模型的方向准确率

In [ ]:
# 使用方法示例：
# 将这段代码放在原始代码的预测部分之后
results = calculate_direction_accuracy(y_test_scaled, preds_scaled)

# 打印详细结果
print("\n=== 方向预测分析 ===")
print(f"总体方向准确率: {results['overall_accuracy']:.2f}%")
print(f"上涨预测准确率: {results['up_accuracy']:.2f}%")
print(f"下跌预测准确率: {results['down_accuracy']:.2f}%")
print(f"总预测次数: {results['total_predictions']}")
print(f"正确预测次数: {results['correct_predictions']}")

print("\n混淆矩阵:")
print(f"正确预测上涨 (True Positive): {results['confusion_matrix']['true_positive']}")
print(f"正确预测下跌 (True Negative): {results['confusion_matrix']['true_negative']}")
print(f"错误预测上涨 (False Positive): {results['confusion_matrix']['false_positive']}")
print(f"错误预测下跌 (False Negative): {results['confusion_matrix']['false_negative']}")

# 可视化方向预测结果
plt.figure(figsize=(14, 7))
plt.subplot(2, 1, 1)
plt.plot(model_data.index[-len(y_test_scaled):], y_test_scaled.flatten(), label='Real Values', color='blue')
plt.plot(model_data.index[-len(preds):], preds.flatten(), label='Predictions', color='orange')
plt.title(f'{ticker} Price Prediction')
plt.legend()
plt.grid(True)

# 添加方向准确性的可视化
plt.subplot(2, 1, 2)
true_direction = results['directions']['true_direction']
pred_direction = results['directions']['pred_direction']
correct_pred = true_direction == pred_direction

plt.scatter(model_data.index[-len(true_direction):][correct_pred],
           [1] * sum(correct_pred),
           color='green',
           label='Correct Direction',
           alpha=0.5)
plt.scatter(model_data.index[-len(true_direction):][~correct_pred],
           [0] * sum(~correct_pred),
           color='red',
           label='Wrong Direction',
           alpha=0.5)
plt.title('Direction Prediction Accuracy')
plt.ylabel('Correct (1) / Wrong (0)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### What would this make on paper? 纸面上能赚多少？

In [ ]:
def paper_backtest(y_true, y_pred, fee=0.0005):
    """Every hour: predicted up -> long, predicted down -> short. Fee 0.05% on each unit traded."""
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    position = np.where(np.diff(y_pred) > 0, 1, -1)
    market = np.diff(y_true) / y_true[:-1]
    before_fees = position * market
    after_fees = before_fees - fee * np.abs(np.diff(np.r_[0, position]))
    print(f"Paper profit before fees: {np.prod(1 + before_fees) - 1:8.1%}")
    print(f"Paper profit after fees:  {np.prod(1 + after_fees) - 1:8.1%}")
    print(f"Buy and hold:             {y_true[-1] / y_true[0] - 1:8.1%}")

In [ ]:
paper_backtest(y_test_scaled, preds_scaled)

## Part 3: Audit 第三部分：审查
Paste the code your agent wrote here, one cell at a time. 把 agent 写的代码粘到这里，一次一个单元格。

### Prompt 2: the fix 修复

In [ ]:
# Paste fixed_cell.py here, then run it.

### Prompt 3: fair comparison 公平比较

In [ ]:
# Paste compare_cell.py here, then run it.

### Prompt 4: a better design 更合理的设计

In [ ]:
# Paste better_cell.py here, then run it.

### Prompt 5: test on data it has never seen 用没见过的数据检验

In [ ]:
# Paste retest_cell.py here, then run it.